# 🚀 KHỐI B-1 (v3): 5-FOLD CNN OUT-OF-FOLD (ÂM TIẾT HÁN NÔM) — THẾ HỆ NHÃN v3
Notebook chạy trên Kaggle GPU (T4 hoặc P100) để huấn luyện 5 fold out-of-fold trên bộ nhãn **v3** (83.239 ô, khoá `nom_idx` thật, bbox v3)
và xuất: `p_visual_oof_v3.csv`, `probs_oof.npz` (đủ vector log-xác suất), `models/fold{0..4}.pt`, `summary.json`, `classes.json`.

Khác bản cũ (`kaggle_run_khoib.ipynb`): dữ liệu `khoib-v3-data` (`KhoiB_v3_kaggle_dataset.zip`), script `train_oof_cnn_v3.py`, kết quả nén thành `p_visual_oof_v3_results.zip`.

**Cài đặt:** Chọn **Settings -> Accelerator -> GPU T4 x2 (hoặc GPU P100)**.

In [ ]:
!nvidia-smi

### 1. Tìm tệp dữ liệu đầu vào (`crops_v3.npz`, `labels_final.csv`, `train_oof_cnn_v3.py`)

In [ ]:
import os, sys, glob, json, hashlib
from pathlib import Path

def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True) + glob.glob(f"**/{name}", recursive=True)
    assert hits, f"Không tìm thấy {name}! Kiểm tra đã Add Input dataset 'khoib-v3-data' chưa."
    return hits[0]

crops_path = find("crops_v3.npz")
labels_path = find("labels_final.csv")
script_path = find("train_oof_cnn_v3.py")
manifest_path = (glob.glob("/kaggle/input/**/MANIFEST.json", recursive=True) or [None])[0]
for p in (crops_path, labels_path, script_path):
    print(f"✓ {p} ({os.path.getsize(p) / (1024*1024):.2f} MB)")

# Đối chiếu md5 với MANIFEST (đúng thế hệ v3)
def md5(p):
    h = hashlib.md5()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""): h.update(chunk)
    return h.hexdigest()
if manifest_path:
    man = json.load(open(manifest_path))
    for name, p in [("crops_v3.npz", crops_path), ("labels_final.csv", labels_path), ("train_oof_cnn_v3.py", script_path)]:
        got = md5(p); exp = man["files"][name]["md5"]
        print(f"  md5 {name}: {'KHỚP' if got == exp else 'LỆCH!'} {got[:8]}")
    print("  git_head lúc đóng gói:", man.get("git_head"))

### 2. Chạy huấn luyện 5-Fold CNN Out-of-fold (B-1, v3)
Thời gian ước tính trên GPU T4 / P100: **~12–15 phút** (15 epoch × 5 fold, ≈56k ô train/fold, ~800 lớp).

In [ ]:
!python {script_path} --crops "{crops_path}" --labels "{labels_path}" --out-dir /kaggle/working/output --epochs 15 --bs 256

### 3. Kiểm tra kết quả đầu ra

In [ ]:
import pandas as pd, numpy as np, json, os

out = "/kaggle/working/output"
csv = f"{out}/p_visual_oof_v3.csv"
for f in ["p_visual_oof_v3.csv", "probs_oof.npz", "summary.json", "classes.json"] + [f"models/fold{k}.pt" for k in range(5)]:
    assert os.path.exists(f"{out}/{f}"), f"Thiếu {f}!"
    print(f"✓ {f} ({os.path.getsize(f'{out}/{f}') / (1024*1024):.2f} MB)")
df_res = pd.read_csv(csv, dtype=str, keep_default_na=False)
print(f"\np_visual_oof_v3.csv: {len(df_res):,} dòng · cột: {list(df_res.columns)}")
display(df_res.head())
z = np.load(f"{out}/probs_oof.npz")
print("probs_oof.npz:", {k: (z[k].shape, str(z[k].dtype)) for k in ["LP", "classes", "fold", "ok"]}, "|", str(z["note"]))
s = json.load(open(f"{out}/summary.json"))
print("\n--- TỔNG KẾT ---")
print(json.dumps({k: s[k] for k in ["n_cells", "n_classes", "fold_formula", "mean_val_top1", "mean_val_top5", "folds", "e2b", "b3_gate_review", "total_time_min"]}, ensure_ascii=False, indent=1))

### 4. Đóng gói kết quả & Tải về máy Mac → giải nén vào `KhoiB/v3/p_visual_oof_v3_results/`

In [ ]:
!cd /kaggle/working/output && zip -r /kaggle/working/p_visual_oof_v3_results.zip p_visual_oof_v3.csv probs_oof.npz summary.json classes.json models
print("\n✅ ĐÃ TẠO FILE NÉN: /kaggle/working/p_visual_oof_v3_results.zip", f"({os.path.getsize('/kaggle/working/p_visual_oof_v3_results.zip') / (1024*1024):.1f} MB)")
from IPython.display import FileLink, display, HTML
display(HTML('<h3>📥 Bấm trực tiếp vào link dưới đây để tải về máy:</h3>'))
display(FileLink('p_visual_oof_v3_results.zip'))